In [ ]:
import yaml
from argparse import Namespace

# load yaml
with open("/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/config/config_preprocess.yaml", "r") as f:
    config = yaml.safe_load(f)

# convert dict -> Namespace
args = Namespace(**config)

print(args)

Namespace(t2_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/t2', dwi_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/dwi', adc_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/adc', output_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed')


In [1]:
import os
import json
import random

In [2]:
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/TCIA_test_data_updated_mask.json', 'r') as f:
    data1 = json.load(f)
    
for i,j in enumerate(data1['test']):
    data1['test'][i]['image'] = os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/t2_registered', j['image'])

with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/TUM_test.json', 'r') as f:
    data2 = json.load(f)
    
for i,j in enumerate(data2['test']):
    data2['test'][i]['image'] = os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/t2_registered', j['image'])

with open('//sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/test_data_updated_with_psa.json', 'r') as f:
    data3 = json.load(f)
    
for i,j in enumerate(data3['test']):
    data3['test'][i]['image'] = os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/COMFORT_data_mpMRI/processed/t2_registered', j['image'])


In [19]:
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/PICAI_cspca_updated_with_psa.json', 'r') as f:
    data4 = json.load(f)

for i,j in enumerate(data4['test']):
    data4['test'][i]['image'] = os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/pirad_model_test_PICAI/processed/t2_registered', j['image'])


In [3]:
data_final = {}
data_final['train'] = data1['test'] #+ data2['test'] #+ data3['test']
random.shuffle(data_final['train'])
data_final['test'] = data3['test']

In [4]:
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/cspca_train_tcia.json', 'w') as f:
    json.dump(data_final, f, indent=4)

In [28]:
import json
import os
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/cspca_train.json', 'r') as f:
    data_final = json.load(f)

In [30]:
for i in data_final['test']:
    
    if 'pirads' in i.keys():
        print('Hello')

[{'image': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/t2_registered/1.3.6.1.4.1.14519.5.2.1.170514610254644270218996322298738651722.nrrd',
  'psa': [6.5, 32.45625027357488],
  'dwi': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/DWI_registered/1.3.6.1.4.1.14519.5.2.1.170514610254644270218996322298738651722.nrrd',
  'adc': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/ADC_clipped/1.3.6.1.4.1.14519.5.2.1.170514610254644270218996322298738651722.nrrd',
  'heatmap': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/heatmaps/1.3.6.1.4.1.14519.5.2.1.170514610254644270218996322298738651722.nrrd',
  'mask': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/prostate_mask/1.3.6.1.4.1.14519.5.2.1.170514610254644270218996322298738651722.nrrd',
  'smooth_mask': '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/smo

In [3]:
import argparse
import logging
import os

import nrrd
import numpy as np
from tqdm import tqdm


def clip_adc(args: argparse.Namespace, adc_min = 0.0, adc_max = 3500.0):

    files = os.listdir(args.adc_dir)
    clip_adc_dir = os.path.join(args.output_dir, "ADC_clipped")
    os.makedirs(clip_adc_dir, exist_ok=True)
    logging.info("Starting clipping ADC")

    for file in tqdm(files):

        adc, header_adc = nrrd.read(os.path.join(args.adc_dir, file))
    
        if np.percentile(adc, 99) < 100:
            adc = adc * 100
 
        adc_clipped = np.clip(adc, adc_min, adc_max)
        adc_normalized = adc_clipped / adc_max
        nrrd.write(os.path.join(clip_adc_dir, file), adc_normalized, header_adc)

    args.adc_dir = clip_adc_dir
    return args


In [4]:
args.adc_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/TCIA_prostate/nrrd_files/processed/ADC_registered'
files = os.listdir(args.adc_dir)
#clip_adc_dir = os.path.join(args.output_dir, "ADC_clipped")
#os.makedirs(clip_adc_dir, exist_ok=True)
#logging.info("Starting clipping ADC")

for file in tqdm(files):

    adc, header_adc = nrrd.read(os.path.join(args.adc_dir, file))

    if np.percentile(adc, 99) < 1000:
        adc = adc * 100
        print('YAY')



 72%|███████▏  | 241/336 [00:05<00:02, 40.09it/s]

YAY


 84%|████████▎ | 281/336 [00:06<00:01, 39.14it/s]

YAY


100%|██████████| 336/336 [00:07<00:00, 43.57it/s]


In [5]:
np.percentile(adc, 99)

3018.0

In [2]:
import argparse
import logging
import os
from pathlib import Path

import yaml

from src.preprocessing.generate_heatmap import get_heatmap
from src.preprocessing.clip_intensity import clip_adc
from src.preprocessing.prostate_mask import get_segmask
from src.preprocessing.register_and_crop import register_files
from src.utils import setup_logging, validate_steps

If you have questions or suggestions, feel free to open an issue at https://github.com/DIAGNijmegen/picai_prep



In [3]:
args.seg_dir = os.path.join(args.output_dir, "prostate_mask")
args.t2_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/t2_registered'
args.dwi_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/DWI_registered'
args.adc_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/ADC_clipped'

In [4]:
import argparse
import logging
import os
import SimpleITK as sitk

import nrrd
import numpy as np
from tqdm import tqdm
from scipy.ndimage import binary_dilation
import shutil

'''
def dilate_prostate_mask_xyz(mask_array, expand_mm=2.0, pixel_spacing_xy=0.4):
    """
    Dilates a 3D prostate mask only in the axial (X,Y) plane.
    Specifically designed for arrays where Z is the LAST dimension (X, Y, Z).
    
    Args:
        mask_array: 3D numpy array of the mask (shape: 232, 232, 23)
        expand_mm: How many millimeters of safety margin you want to add
        pixel_spacing_xy: The in-plane spacing (e.g., 0.4 mm)
    """
    # 1. Calculate how many pixels to expand radially
    iterations = int(np.round(expand_mm / pixel_spacing_xy))
    
    # 2. Create the structuring element for (X, Y, Z)
    structuring_element = np.zeros((3, 3, 3), dtype=bool)
    
    # We set the middle Z-index (index 1) to True across all X and Y.
    # This forces the math to spread outward in X and Y, but never up/down in Z.
    structuring_element[:, :, 1] = True  
    
    # 3. Apply the dilation
    dilated_mask = binary_dilation(mask_array > 0, 
                                   structure=structuring_element, 
                                   iterations=iterations)
    
    # 4. Return as the original integer type
    return dilated_mask.astype(mask_array.dtype)
'''

def smoothen_mask(args, sigma=1.0):
    
    files = os.listdir(args.seg_dir)
    
    for file in files:
    
        mask = sitk.ReadImage(os.path.join(args.seg_dir, file))
        mask = sitk.Cast(mask, sitk.sitkFloat32)

        # Smooth
        smoothed = sitk.DiscreteGaussian(mask, variance=1.5)

        # Threshold back to binary
        smooth_mask = smoothed > 0.5
        smooth_mask = sitk.Cast(smooth_mask, sitk.sitkUInt8)
        
        out_path = os.path.join(args.smooth_seg_dir_temp, file)
        sitk.WriteImage(smooth_mask, out_path)




files = os.listdir(args.t2_dir)
args.heatmapdir = os.path.join(args.output_dir, "heatmaps/")
os.makedirs(args.heatmapdir, exist_ok=True)
args.smooth_seg_dir_temp = os.path.join(args.output_dir, "smooth_prostate_mask_temp/")
os.makedirs(args.smooth_seg_dir_temp, exist_ok=True)
args.smooth_seg_dir = os.path.join(args.output_dir, "smooth_prostate_mask/")
os.makedirs(args.smooth_seg_dir, exist_ok=True)
smoothen_mask(args)



In [ ]:
print(file

'206753830.nrrd'

In [5]:
for file in tqdm(files):
    bool_dwi = False
    bool_adc = False
    mask_temp, header_mask = nrrd.read(os.path.join(args.seg_dir, file))
    #spacing = np.linalg.norm(header_mask['space directions'], axis=1)
    dwi, header_dwi = nrrd.read(os.path.join(args.dwi_dir, file))
    adc, header_adc = nrrd.read(os.path.join(args.adc_dir, file))
    nonzero_vals_dwi = dwi[mask_temp > 0]
    mask, _ = nrrd.read(os.path.join(args.smooth_seg_dir_temp, file))
    mask = np.maximum(mask, mask_temp)

    if len(nonzero_vals_dwi) > 0:
        #min_val = nonzero_vals_dwi.min()
        #max_val = nonzero_vals_dwi.max()
        min_val, max_val = np.percentile(nonzero_vals_dwi, [1, 99])
        clipped_dwi = np.clip(dwi, min_val, max_val)


        heatmap_dwi = np.zeros_like(clipped_dwi, dtype=np.float32)

        if min_val != max_val:
            heatmap_dwi = (clipped_dwi - min_val) / (max_val - min_val)
            masked_heatmap_dwi = np.where(mask > 0, heatmap_dwi, 0)
        else:
            bool_dwi = True

    else:
        bool_dwi = True

    nonzero_vals_adc = adc[mask > 0]

    if len(nonzero_vals_adc) > 0:
        min_val = 0
        max_val = 1
        heatmap_adc = np.zeros_like(adc, dtype=np.float32)
        heatmap_adc = (max_val - adc) / (max_val - min_val)
        masked_heatmap_adc = np.where(mask > 0, heatmap_adc, 0)


    else:
        bool_adc = True

    if not bool_dwi and not bool_adc:
        mix_mask = (masked_heatmap_dwi * 0.3 ) + (masked_heatmap_adc * 0.7)
        #mix_mask = (masked_heatmap_dwi**0.5) * (masked_heatmap_adc**2.0)
        write_header = header_dwi
    elif bool_dwi:
        mix_mask = masked_heatmap_adc
        write_header = header_adc
    else:
        mix_mask = np.ones_like(adc, dtype=np.float32)
        write_header = header_dwi

    mix_mask = (mix_mask - mix_mask[mask>0].min()) / (mix_mask[mask>0].max() - mix_mask[mask>0].min())
    mix_mask =  np.where(mask > 0, mix_mask, 0)
    nrrd.write(os.path.join(args.heatmapdir, file), mix_mask, write_header)
    nrrd.write(os.path.join(args.smooth_seg_dir, file), mask, header_mask)

shutil.rmtree(args.smooth_seg_dir_temp)



 12%|█▏        | 13/112 [00:02<00:19,  5.08it/s]


ValueError: zero-size array to reduction operation minimum which has no identity

In [2]:
import argparse
import logging
import os

import nrrd
import numpy as np
import torch
from monai.bundle import ConfigParser
from monai.data import MetaTensor
from monai.transforms import (
    Compose,
    EnsureChannelFirstd,
    EnsureTyped,
    LoadImaged,
    NormalizeIntensityd,
    Orientationd,
    ScaleIntensityd,
    Spacingd,
)
from monai.utils import set_determinism
from tqdm import tqdm

set_determinism(43)

def is_continuous(idx):
    idx = sorted(idx)
    return all(b - a == 1 for a, b in zip(idx, idx[1:]))

"""
Generate prostate segmentation masks using a pre-trained deep learning model.
This function performs inference on T2-weighted MRI images to segment the prostate gland.
It applies preprocessing transformations, runs the segmentation model, and saves the
predicted masks. Post-processing is applied to retain only the top 10 slices with
the highest non-zero voxel counts.
Args:
    args: An arguments object containing:
        - output_dir (str): Base output directory where segmentation masks will be saved
        - project_dir (str): Root project directory containing model config and checkpoint
        - t2_dir (str): Directory containing input T2-weighted MRI images in NRRD format
Returns:
    args: The updated arguments object with seg_dir added, pointing to the
            prostate_mask subdirectory within output_dir
Raises:
    FileNotFoundError: If the model checkpoint or config file is not found
    RuntimeError: If CUDA operations fail on GPU
Notes:
    - Automatically selects GPU (CUDA) if available, otherwise uses CPU
    - Applies MONAI transformations: loading, orientation (RAS), spacing (0.5mm isotropic),
        intensity scaling and normalization
    - Post-processing filters predictions to top 10 slices by non-zero voxel density
    - Output masks are saved in NRRD format preserving original image headers
"""

args.seg_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/og_mask'
os.makedirs(args.seg_dir, exist_ok=True)
args.project_dir = '/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate'
args.t2_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/t2_registered'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
model_config_file = os.path.join(args.project_dir, "config", "inference.json")
model_config = ConfigParser()
model_config.read_config(model_config_file)
model_config["output_dir"] = args.seg_dir
model_config["dataset_dir"] = args.t2_dir
files = os.listdir(args.t2_dir)
model_config["datalist"] = [os.path.join(args.t2_dir, f) for f in files]
checkpoint = os.path.join(
    args.project_dir,
    "models",
    "prostate_segmentation_model.pt",
)
model = model_config.get_parsed_content("network_def").to(device)
inferer = model_config.get_parsed_content("inferer")
model.load_state_dict(torch.load(checkpoint, map_location=device))
model.eval()

keys = "image"
transform = Compose(
    [
        LoadImaged(keys=keys),
        EnsureChannelFirstd(keys=keys),
        Orientationd(keys=keys, axcodes="RAS"),
        Spacingd(keys=keys, pixdim=[0.5, 0.5, 0.5], mode="bilinear"),
        ScaleIntensityd(keys=keys, minv=0, maxv=1),
        NormalizeIntensityd(keys=keys),
        EnsureTyped(keys=keys),
    ]
)
logging.info("Starting prostate segmentation")
for file in tqdm(files):
    data = {"image": os.path.join(args.t2_dir, file)}
    _, header_t2 = nrrd.read(data["image"])
    transformed_data = transform(data)
    a = transformed_data
    with torch.no_grad():
        images = a["image"].reshape(1, *(a["image"].shape)).to(device)
        data["pred"] = inferer(images, network=model)
    pred_img = data["pred"].argmax(1).cpu()

    model_output = {}
    model_output["image"] = MetaTensor(pred_img, meta=transformed_data["image"].meta)
    transformed_data["image"].data = model_output["image"].data
    temp = transform.inverse(transformed_data)
    pred_temp = temp["image"][0].numpy()
    pred_nrrd = np.round(pred_temp)

    nonzero_counts = np.count_nonzero(pred_nrrd, axis=(0, 1))
    top_slices = np.argsort(nonzero_counts)

    output_ = np.zeros_like(pred_nrrd)
    output_[:, :, top_slices] = pred_nrrd[:, :, top_slices]

    nrrd.write(os.path.join(args.seg_dir, file), pred_nrrd, header_t2)



You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
100%|██████████| 111/111 [01:09<00:00,  1.59it/s]


In [2]:
import argparse
import logging
import os
from pathlib import Path

import yaml

from src.preprocessing.generate_heatmap import get_heatmap
from src.preprocessing.clip_intensity import clip_adc
from src.preprocessing.prostate_mask import get_segmask
from src.preprocessing.register_and_crop import register_files
from src.utils import setup_logging, validate_steps

If you have questions or suggestions, feel free to open an issue at https://github.com/DIAGNijmegen/picai_prep



In [11]:
args

Namespace(t2_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/t2', dwi_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/dwi', adc_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/adc', output_dir='/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed')

In [3]:
args.t2_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/t2_registered'
args.dwi_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/DWI_registered'
args.adc_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/ADC_registered'
args.margim = 0.2
args.project_dir = '/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate'
args.output_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed'
args.seg_dir = '/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/cropped_prostate_mask'


In [17]:
args = clip_adc(args)

100%|██████████| 6733/6733 [08:27<00:00, 13.26it/s]


In [15]:
args.adc_dir

'/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/ADC_clipped'

In [ ]:
args = get_heatmap(args)

In [3]:
import nrrd
from AIAH_utility.viewer import BasicViewer

In [ ]:
files = os.listdir('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/prostate_mask')
BasicViewer(nrrd.read(os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/prostate_mask', files[0]))[0]).show()

In [5]:
BasicViewer(nrrd.read(os.path.join('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate-foundation/processed/cropped_prostate_mask', files[0]))[0]).show()

In [3]:
import json
import os
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/TUM_test.json', 'r') as f:
    data_list = json.load(f)

In [4]:
files = os.listdir('/sc-projects/sc-proj-cc06-ag-ki-radiologie/prostate_test/TUM/data_prostate_test/processed/t2_registered')
data_list_new = [i for i in data_list['test'] if i['image'] in files]


In [5]:
data_list['test'] = data_list_new

In [6]:
with open('/sc-scratch/sc-scratch-cc06-ag-ki-radiologie/prostate_foundation/WSAttention-Prostate/dataset/TUM_test_updated.json', 'w') as f:
    json.dump(data_list, f, indent=4)